In [3]:
import pandas as pd
import geopandas as gdp
import ee
import geemap
import os
from pathlib import Path


/Users/esthergreenwood/Documents/GitHub/mapping_sanitation_access_LMICs/.venv/lib/python3.9/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/esthergreenwood/Documents/GitHub/mapping_sanitation_access_LMICs/.venv/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/esthergreenwood/Documents/GitHub/mapping_sanitation_access_LMICs/.venv/lib/python3.9/site-p

In [11]:

ee.Authenticate(auth_mode='localhost')

True

In [16]:
ee.Initialize(project="ee-estheregreenwood")

In [4]:
# DHS_boundary data are in directory with per country subdirectories containing the DHS region shapefiles (in subdir 'shps')
# MICS_boundary data are in directory with per country subdirectories containing the MICS region shapefiles (in subdir 'Shapefiles'); one shapefile per admin level

DHS_boundaries = 'data/DHS_boundaries'
MICS_boundaries = 'data/MICS_boundaries'

# Build allowed (country, year) set from download filter CSV
# Directories use either start or end year of "YYYY-YY" ranges, so we expand both
import re

def years_from_csv_year(year_str):
    year_str = str(year_str).strip()
    years = set(int(y) for y in re.findall(r'\d{4}', year_str))
    m = re.match(r'(\d{4})-(\d{2})$', year_str)
    if m:
        start = int(m.group(1))
        years.add((start // 100) * 100 + int(m.group(2)))
    return years

dhs_filter = pd.read_csv('data/P04-00-DHS_Boundaries_v1.0.csv', encoding='utf-8-sig')
dhs_filter = dhs_filter[dhs_filter['Download Boundaries'] == 1]
included_dhs_studies = set()
for _, row in dhs_filter.iterrows():
    for yr in years_from_csv_year(row['Year']):
        included_dhs_studies.add((row['Country Name'], yr))

# Load all DHS shapefiles
dhs_gdfs = []
for country_dir in Path(DHS_boundaries).iterdir():
    shp = country_dir / 'shps' / 'sdr_subnational_boundaries.shp'
    if shp.exists():
        # Directory format: "Country Name YYYY TYPE"
        parts = country_dir.name.rsplit(' ', 2)
        dir_country, dir_year = ' '.join(parts[:-2]), int(parts[-2])
        if (dir_country, dir_year) in included_dhs_studies:
            dhs_gdfs.append(gdp.read_file(shp))
dhs_gdf = gdp.GeoDataFrame(pd.concat(dhs_gdfs, ignore_index=True))

# Load all MICS shapefiles — track admin_level from filename suffix (1/2/3)
mics_gdfs = []
for country_dir in Path(MICS_boundaries).iterdir():
    shapefiles_dir = country_dir / 'Shapefiles'
    if shapefiles_dir.exists():
        for shp in sorted(shapefiles_dir.glob('mics_boundaries*.shp')):
            suffix = shp.stem.replace('mics_boundaries', '')
            level = int(suffix) if suffix.isdigit() else 1
            gdf = gdp.read_file(shp)
            gdf['admin_level'] = level
            mics_gdfs.append(gdf)
import warnings
with warnings.catch_warnings():
    warnings.simplefilter('ignore', FutureWarning)
    mics_gdf = gdp.GeoDataFrame(pd.concat([gdf for gdf in mics_gdfs if not gdf.empty], ignore_index=True))

# Drop null/empty geometries and force 2D
dhs_gdf = dhs_gdf[~(dhs_gdf.geometry.isna() | dhs_gdf.geometry.is_empty)].copy()
dhs_gdf['geometry'] = dhs_gdf['geometry'].force_2d()
mics_gdf = mics_gdf[~(mics_gdf.geometry.isna() | mics_gdf.geometry.is_empty)].copy()
mics_gdf['geometry'] = mics_gdf['geometry'].force_2d()

# Add analysis year column to both dataframes (DHS uses 'SVYYEAR', MICS uses 'SVYYEARS')
dhs_gdf['analysis_year'] = dhs_gdf['SVYYEAR'].astype(int)
dhs_gdf['admin_level'] = 1  # DHS always one level per survey
mics_gdf['analysis_year'] = mics_gdf['SVYYEARS']

print('DHS size:', len(dhs_gdf))
print('MICS size:', len(mics_gdf))

FileNotFoundError: [Errno 2] No such file or directory: 'data/P04-00-DHS_Boundaries_v1.0.csv'

In [17]:
# Defining adminstrative areas reported in household surveys
# Define island groups for Kiribati
noGil = ee.Geometry.Polygon(
        [[[173.76214858993802, 1.688437284048748],
          [173.89398452743802, 3.2689380495714633],
          [172.53167983993802, 4.277476168442425],
          [172.05246629783107, 3.029716351603881],
          [172.84548591405644, 1.4656846206052414],
          [173.12175664737128, 1.3820186787491144],
          [173.1354466421833, 1.3865663776048271],
          [173.14021024539375, 1.3954043333854713]]], None, False)
souTara = ee.Geometry.Polygon(
    [[[173.14024407183683, 1.3954382342837202],
        [173.13552338397062, 1.3865573758520389],
        [173.12591034686125, 1.382867734037598],
        [172.9929535267801, 1.3664672516799288],
        [172.8940765736551, 1.3946115454160364],
        [172.8940765736551, 1.317042332174653],
        [173.04582523088166, 1.304685947700206],
        [173.21679996232697, 1.3609756435068756],
        [173.17560123185822, 1.406967470897869]]], None, False)
cenGil = ee.Geometry.Polygon(
    [[[172.75257236267254, 1.106559466163889],
        [172.75257236267254, -0.947771870448704],
        [174.70813876892254, -0.947771870448704],
        [174.70813876892254, 1.106559466163889]]], None, False)
souGil = ee.Geometry.Polygon(
    [[[174.44446689392254, -1.0356493216635867],
        [174.44446689392254, -2.858249945133194],
        [177.15808994079754, -2.858249945133194],
        [177.15808994079754, -1.0356493216635867]]], None, False)
lineAndPhoenix =  ee.Geometry.Polygon(
    [[[178.7565749612447, 8.140225397293124],
        [178.7565749612447, -20.88502241263751],
        [223.75657496124472, -20.88502241263751],
        [223.75657496124472, 8.140225397293124]]], None, False)

# Define the countries to sample from level 0, 2, 3 or 1
countryLevel0 = ['Kiribati','Tonga']
countryLevel1 = ['Gambia','Iraq','Togo', 'Chad','Georgia','Ghana','Guinea-Bissau','Guyana','Laos','Lesotho','Nigeria','Paraguay','Sierra Leone','Suriname','Tonga','Zimbabwe']
countryLevel2 = ['Bangladesh','Gambia','Iraq','Kosovo','Madagascar','Palestina','Togo', 'São Tomé and Príncipe']
countryLevel3 = ['Pakistan']

# Load the GADM datasets 
GADM_level0 = ee.FeatureCollection('users/crowtherlab/GADM36/GADM36_level0') \
                    .filter(ee.Filter.inList('NAME_0',['Kiribati']))

GADM_level1_prep = ee.FeatureCollection('users/crowtherlab/GADM36/GADM36_level1')

GADM_level1 = GADM_level1_prep.filter(ee.Filter.inList('NAME_0',countryLevel1)) \
                                  .map(lambda f: f.set('HH7',f.getString('NAME_1')))

GADM_level2_prep = ee.FeatureCollection('users/crowtherlab/GADM36/GADM36_level2')
GADM_level2 = GADM_level2_prep.filter(ee.Filter.inList('NAME_0',countryLevel2)) \
                                  .map(lambda f: f.set('HH7',f.getString('NAME_2')))
                    
GADM_level3_prep = ee.FeatureCollection('users/crowtherlab/GADM36/GADM36_level3')
GADM_level3 = GADM_level3_prep \
                    .filter(ee.Filter.inList('NAME_0',countryLevel3)) \
                    .map(lambda f: f.set('HH7',f.getString('NAME_3')))

# Define Kiribati 
NORTHERN_GILBERT = ee.Feature(GADM_level0.geometry().intersection(noGil, 10)).set('NAME_1','NORTHERN GILBERT').set('NAME_0','Kiribati').set('HH7','NORTHERN_GILBERT')
SOUTH_TARAWA = ee.Feature(GADM_level0.geometry().intersection(souTara, 10)).set('NAME_1','SOUTH TARAWA').set('NAME_0','Kiribati').set('HH7','SOUTH_TARAWA')
CENTRAL_GILBERT = ee.Feature(GADM_level0.geometry().intersection(cenGil, 10)).set('NAME_1','CENTRAL GILBERT').set('NAME_0','Kiribati').set('HH7','CENTRAL_GILBERT')
SOUTHERN_GILBERT = ee.Feature(GADM_level0.geometry().intersection(souGil, 10)).set('NAME_1','SOUTHERN GILBERT').set('NAME_0','Kiribati').set('HH7','SOUTHERN_GILBERT')
LINE_AND_PHOENIX = ee.Feature(GADM_level0.geometry().intersection(lineAndPhoenix, 10)).set('NAME_1','LINE AND PHOENIX GROUP').set('NAME_0','Kiribati').set('HH7','LINE_AND_PHOENIX')
GADM_level0 = ee.FeatureCollection([NORTHERN_GILBERT, SOUTH_TARAWA, CENTRAL_GILBERT, SOUTHERN_GILBERT, LINE_AND_PHOENIX])

##/ Define the new regions for Level 1
# Algeria 
NORD_CENTRE = ['Alger','Blida','Boumerdès','Tipaza','Bouira','Médéa','Tizi Ouzou','Béjaïa','Chlef','Aïn Defla']
NORD_EST = ['Annaba','Constantine','Skikda','Jijel','Mila','Souk Ahras','El Tarf','Guelma']
NORD_OUEST = ['Oran','Tlemcen','Mostaganem','Aïn Témouchent','Relizane','Sidi Bel Abbès','Mascara']
HAUT_PLATEAU_CENTRE = ['Djelfa','Laghouat',"M'Sila"]
HAUT_PLATEAU_EST = ['Sétif','Batna','Khenchela','Bordj Bou Arréridj','Oum el Bouaghi','Tébessa']
HAUT_PLATEAU_OUEST = ['Saïda','Tissemsilt','Naâma','El Bayadh','Tiaret']
SUD = ['Béchar','Tindouf','Adrar','Ghardaïa','Biskra','El Oued','Ouargla','Tamanghasset','Illizi']

# Central African Republic 
REGION_1 = ["Ombella-M'Poko","Lobaye"]
REGION_2 = ["Mambéré-Kadéï","Nana-Mambéré","Sangha-Mbaéré"]
REGION_3 = ["Ouham-Pendé","Ouham"]
REGION_4 = ["Kémo","Nana-Grébizi","Ouaka"]
REGION_5 = ["Bamingui-Bangoran","Haute-Kotto","Vakaga"]
REGION_6 = ["Basse-Kotto","Mbomou","Haut-Mbomou"]
REGION_7 = ["Bangui"]

# Mongolia
WESTERN = ["Bayan-Ölgiy","Govi-Altay","Dzavhan","Hovd","Uvs"]
EASTERN = ["Dornod","Sühbaatar","Hentiy"]
ULAANBAATAR = ["Ulaanbaatar"]
CENTRAL = ["Töv","Ömnögovi","Selenge","Orhon","Darhan-Uul","Govisümber","Dundgovi","Dornogovi"]
KHANGAI = ["Bayanhongor","Hövsgöl","Arhangay","Övörhangay","Bulgan"]

# Tunisia
TUNISIA_District_Tunis = ["Ariana","Ben Arous (Tunis Sud)","Bizerte","Manubah","Nabeul","Tunis","Zaghouan"]
TUNISIA_NORD_OUEST = ["Béja","Jendouba","Le Kef","Siliana"]
TUNISIA_CENTRE_EST = ["Mahdia","Monastir","Sousse"]
TUNISIA_CENTRE_OUEST = ["Kairouan","Kassérine","Sidi Bou Zid"]
TUNISIA_SUD_EST = ["Gabès","Médenine","Sfax","Tataouine"]
TUNISIA_SUD_OUEST = ["Gafsa","Kebili","Tozeur"]

# Concatenate the districts into regions
NORD_CENTRE = GADM_level1_prep.filterMetadata('NAME_0','equals','Algeria').filter(ee.Filter.inList("NAME_1",NORD_CENTRE))
NORD_EST = GADM_level1_prep.filterMetadata('NAME_0','equals','Algeria').filter(ee.Filter.inList("NAME_1",NORD_EST))
NORD_OUEST = GADM_level1_prep.filterMetadata('NAME_0','equals','Algeria').filter(ee.Filter.inList("NAME_1",NORD_OUEST))
HAUT_PLATEAU_CENTRE = GADM_level1_prep.filterMetadata('NAME_0','equals','Algeria').filter(ee.Filter.inList("NAME_1",HAUT_PLATEAU_CENTRE))
HAUT_PLATEAU_EST = GADM_level1_prep.filterMetadata('NAME_0','equals','Algeria').filter(ee.Filter.inList("NAME_1",HAUT_PLATEAU_EST))
HAUT_PLATEAU_OUEST = GADM_level1_prep.filterMetadata('NAME_0','equals','Algeria').filter(ee.Filter.inList("NAME_1",HAUT_PLATEAU_OUEST))
SUD = GADM_level1_prep.filterMetadata('NAME_0','equals','Algeria').filter(ee.Filter.inList("NAME_1",SUD))

REGION_1 = GADM_level1_prep.filterMetadata('NAME_0','equals','Central African Republic').filter(ee.Filter.inList("NAME_1",REGION_1))
REGION_2 = GADM_level1_prep.filterMetadata('NAME_0','equals','Central African Republic').filter(ee.Filter.inList("NAME_1",REGION_2))
REGION_3 = GADM_level1_prep.filterMetadata('NAME_0','equals','Central African Republic').filter(ee.Filter.inList("NAME_1",REGION_3))
REGION_4 = GADM_level1_prep.filterMetadata('NAME_0','equals','Central African Republic').filter(ee.Filter.inList("NAME_1",REGION_4))
REGION_5 = GADM_level1_prep.filterMetadata('NAME_0','equals','Central African Republic').filter(ee.Filter.inList("NAME_1",REGION_5))
REGION_6 = GADM_level1_prep.filterMetadata('NAME_0','equals','Central African Republic').filter(ee.Filter.inList("NAME_1",REGION_6))
REGION_7 = GADM_level1_prep.filterMetadata('NAME_0','equals','Central African Republic').filter(ee.Filter.inList("NAME_1",REGION_7))

WESTERN = GADM_level1_prep.filterMetadata('NAME_0','equals','Mongolia').filter(ee.Filter.inList("NAME_1",WESTERN))
EASTERN = GADM_level1_prep.filterMetadata('NAME_0','equals','Mongolia').filter(ee.Filter.inList("NAME_1",EASTERN))
ULAANBAATAR = GADM_level1_prep.filterMetadata('NAME_0','equals','Mongolia').filter(ee.Filter.inList("NAME_1",ULAANBAATAR))
CENTRAL = GADM_level1_prep.filterMetadata('NAME_0','equals','Mongolia').filter(ee.Filter.inList("NAME_1",CENTRAL))
KHANGAI = GADM_level1_prep.filterMetadata('NAME_0','equals','Mongolia').filter(ee.Filter.inList("NAME_1",KHANGAI))
           
TUNISIA_District_Tunis = GADM_level1_prep.filterMetadata('NAME_0','equals','Tunisia').filter(ee.Filter.inList("NAME_1",TUNISIA_District_Tunis))
TUNISIA_NORD_OUEST = GADM_level1_prep.filterMetadata('NAME_0','equals','Tunisia').filter(ee.Filter.inList("NAME_1",TUNISIA_NORD_OUEST))
TUNISIA_CENTRE_EST = GADM_level1_prep.filterMetadata('NAME_0','equals','Tunisia').filter(ee.Filter.inList("NAME_1",TUNISIA_CENTRE_EST))
TUNISIA_CENTRE_OUEST = GADM_level1_prep.filterMetadata('NAME_0','equals','Tunisia').filter(ee.Filter.inList("NAME_1",TUNISIA_CENTRE_OUEST))
TUNISIA_SUD_EST = GADM_level1_prep.filterMetadata('NAME_0','equals','Tunisia').filter(ee.Filter.inList("NAME_1",TUNISIA_SUD_EST))
TUNISIA_SUD_OUEST = GADM_level1_prep.filterMetadata('NAME_0','equals','Tunisia').filter(ee.Filter.inList("NAME_1",TUNISIA_SUD_OUEST))
                          
# Final FCs
Algeria = ee.FeatureCollection([
    ee.Feature(NORD_CENTRE.union().geometry()).set('NAME_0','Algeria').set('NAME_1','NORD CENTRE').set('NAME_1s',NORD_CENTRE.aggregate_array('NAME_1').join('|')).set('GID_1s',NORD_CENTRE.aggregate_array('GID_1').join('|')).set('HH7','NORD_CENTRE'),
    ee.Feature(NORD_EST.union().geometry()).set('NAME_0','Algeria').set('NAME_1','NORD EST').set('NAME_1s',NORD_EST.aggregate_array('NAME_1').join('|')).set('GID_1s',NORD_EST.aggregate_array('GID_1').join('|')).set('HH7','NORD_EST'),
    ee.Feature(NORD_OUEST.union().geometry()).set('NAME_0','Algeria').set('NAME_1','NORD OUEST').set('NAME_1s',NORD_OUEST.aggregate_array('NAME_1').join('|')).set('GID_1s',NORD_OUEST.aggregate_array('GID_1').join('|')).set('HH7','NORD_OUEST'),
    ee.Feature(HAUT_PLATEAU_CENTRE.union().geometry()).set('NAME_0','Algeria').set('NAME_1','HAUT PLATEAU CENTRE').set('NAME_1s',HAUT_PLATEAU_CENTRE.aggregate_array('NAME_1').join('|')).set('GID_1s',HAUT_PLATEAU_CENTRE.aggregate_array('GID_1').join('|')).set('HH7','HAUT_PLATEAU_CENTRE'),
    ee.Feature(HAUT_PLATEAU_EST.union().geometry()).set('NAME_0','Algeria').set('NAME_1','HAUT PLATEAU EST').set('NAME_1s',HAUT_PLATEAU_EST.aggregate_array('NAME_1').join('|')).set('GID_1s',HAUT_PLATEAU_EST.aggregate_array('GID_1').join('|')).set('HH7','HAUT_PLATEAU_EST'),
    ee.Feature(HAUT_PLATEAU_OUEST.union().geometry()).set('NAME_0','Algeria').set('NAME_1','HAUT PLATEAU OUEST').set('NAME_1s',HAUT_PLATEAU_OUEST.aggregate_array('NAME_1').join('|')).set('GID_1s',HAUT_PLATEAU_OUEST.aggregate_array('GID_1').join('|')).set('HH7','HAUT_PLATEAU_OUEST'),
    ee.Feature(SUD.union().geometry()).set('NAME_0','Algeria').set('NAME_1','SUD').set('NAME_1s',SUD.aggregate_array('NAME_1').join('|')).set('GID_1s',SUD.aggregate_array('GID_1').join('|')).set('HH7','SUD'),
    ])
CAR = ee.FeatureCollection([
    ee.Feature(REGION_1.union().geometry()).set('NAME_0','Central African Republic').set('NAME_1','REGION_1').set('NAME_1s',REGION_1.aggregate_array('NAME_1').join('|')).set('GID_1s',REGION_1.aggregate_array('GID_1').join('|')).set("HH7","REGION_1"),
    ee.Feature(REGION_2.union().geometry()).set('NAME_0','Central African Republic').set('NAME_1','REGION_2').set('NAME_1s',REGION_2.aggregate_array('NAME_1').join('|')).set('GID_1s',REGION_2.aggregate_array('GID_1').join('|')).set("HH7","REGION_2"),
    ee.Feature(REGION_3.union().geometry()).set('NAME_0','Central African Republic').set('NAME_1','REGION_3').set('NAME_1s',REGION_3.aggregate_array('NAME_1').join('|')).set('GID_1s',REGION_3.aggregate_array('GID_1').join('|')).set("HH7","REGION_3"),
    ee.Feature(REGION_4.union().geometry()).set('NAME_0','Central African Republic').set('NAME_1','REGION_4').set('NAME_1s',REGION_4.aggregate_array('NAME_1').join('|')).set('GID_1s',REGION_4.aggregate_array('GID_1').join('|')).set("HH7","REGION_4"),
    ee.Feature(REGION_5.union().geometry()).set('NAME_0','Central African Republic').set('NAME_1','REGION_5').set('NAME_1s',REGION_5.aggregate_array('NAME_1').join('|')).set('GID_1s',REGION_5.aggregate_array('GID_1').join('|')).set("HH7","REGION_5"),
    ee.Feature(REGION_6.union().geometry()).set('NAME_0','Central African Republic').set('NAME_1','REGION_6').set('NAME_1s',REGION_6.aggregate_array('NAME_1').join('|')).set('GID_1s',REGION_6.aggregate_array('GID_1').join('|')).set("HH7","REGION_6"),
    ee.Feature(REGION_7.union().geometry()).set('NAME_0','Central African Republic').set('NAME_1','REGION_7').set('NAME_1s',REGION_7.aggregate_array('NAME_1').join('|')).set('GID_1s',REGION_7.aggregate_array('GID_1').join('|')).set("HH7","REGION_7")
    ])
Mongolia = ee.FeatureCollection([
    ee.Feature(WESTERN.union().geometry()).set('NAME_0','Mongolia').set('NAME_1','WESTERN').set('NAME_1s',WESTERN.aggregate_array('NAME_1').join('|')).set('GID_1s',WESTERN.aggregate_array('GID_1').join('|')).set('HH7','WESTERN'),
    ee.Feature(EASTERN.union().geometry()).set('NAME_0','Mongolia').set('NAME_1','EASTERN').set('NAME_1s',EASTERN.aggregate_array('NAME_1').join('|')).set('GID_1s',EASTERN.aggregate_array('GID_1').join('|')).set('HH7','EASTERN'),
    ee.Feature(ULAANBAATAR.union().geometry()).set('NAME_0','Mongolia').set('NAME_1','ULAANBAATAR').set('NAME_1s',ULAANBAATAR.aggregate_array('NAME_1').join('|')).set('GID_1s',ULAANBAATAR.aggregate_array('GID_1').join('|')).set('HH7','ULAANBAATAR'),
    ee.Feature(CENTRAL.union().geometry()).set('NAME_0','Mongolia').set('NAME_1','CENTRAL').set('NAME_1s',CENTRAL.aggregate_array('NAME_1').join('|')).set('GID_1s',CENTRAL.aggregate_array('GID_1').join('|')).set('HH7','CENTRAL'),
    ee.Feature(KHANGAI.union().geometry()).set('NAME_0','Mongolia').set('NAME_1','KHANGAI').set('NAME_1s',KHANGAI.aggregate_array('NAME_1').join('|')).set('GID_1s',KHANGAI.aggregate_array('GID_1').join('|')).set('HH7','KHANGAI')
    ])
Tunisia = ee.FeatureCollection([
    ee.Feature(TUNISIA_District_Tunis.union().geometry()).set('NAME_0','Tunisia').set('NAME_1','TUNISIA_District_Tunis').set('NAME_1s',TUNISIA_District_Tunis.aggregate_array('NAME_1').join('|')).set('GID_1s',TUNISIA_District_Tunis.aggregate_array('GID_1').join('|')).set('HH7','TUNISIA_District_Tunis'),
    ee.Feature(TUNISIA_NORD_OUEST.union().geometry()).set('NAME_0','Tunisia').set('NAME_1','TUNISIA_NORD_OUEST').set('NAME_1s',TUNISIA_NORD_OUEST.aggregate_array('NAME_1').join('|')).set('GID_1s',TUNISIA_NORD_OUEST.aggregate_array('GID_1').join('|')).set('HH7','TUNISIA_NORD_OUEST'),
    ee.Feature(TUNISIA_CENTRE_EST.union().geometry()).set('NAME_0','Tunisia').set('NAME_1','TUNISIA_CENTRE_EST').set('NAME_1s',TUNISIA_CENTRE_EST.aggregate_array('NAME_1').join('|')).set('GID_1s',TUNISIA_CENTRE_EST.aggregate_array('GID_1').join('|')).set('HH7','TUNISIA_CENTRE_EST'),
    ee.Feature(TUNISIA_CENTRE_OUEST.union().geometry()).set('NAME_0','Tunisia').set('NAME_1','TUNISIA_CENTRE_OUEST').set('NAME_1s',TUNISIA_CENTRE_OUEST.aggregate_array('NAME_1').join('|')).set('GID_1s',TUNISIA_CENTRE_OUEST.aggregate_array('GID_1').join('|')).set('HH7','TUNISIA_CENTRE_OUEST'),
    ee.Feature(TUNISIA_SUD_EST.union().geometry()).set('NAME_0','Tunisia').set('NAME_1','TUNISIA_SUD_EST').set('NAME_1s',TUNISIA_SUD_EST.aggregate_array('NAME_1').join('|')).set('GID_1s',TUNISIA_SUD_EST.aggregate_array('GID_1').join('|')).set('HH7','TUNISIA_SUD_EST'),
    ee.Feature(TUNISIA_SUD_OUEST.union().geometry()).set('NAME_0','Tunisia').set('NAME_1','TUNISIA_SUD_OUEST').set('NAME_1s',TUNISIA_SUD_OUEST.aggregate_array('NAME_1').join('|')).set('GID_1s',TUNISIA_SUD_OUEST.aggregate_array('GID_1').join('|')).set('HH7','TUNISIA_SUD_OUEST')
    ])    
GADM_level1 = GADM_level1.merge(Algeria).merge(CAR).merge(Tunisia).merge(Mongolia)

# Add the missing properties 
GADM_level3 = GADM_level3.map(lambda f: f.set('NAME_1s', 'NaN').set('GID_1s', 'NaN'))
GADM_level0 = GADM_level0.map(lambda f: f.set(ee.Dictionary.fromLists(                                                                
      p := GADM_level3.first().propertyNames().removeAll(GADM_level0.first().propertyNames()),                                          
      ee.List.repeat('NaN', p.size())                                                                                                   
  )))             
GADM_level2 = GADM_level2.map(lambda f: f.set(ee.Dictionary.fromLists(                                                                
      p := GADM_level3.first().propertyNames().removeAll(GADM_level2.first().propertyNames()),
      ee.List.repeat('NaN', p.size())                                                                                                   
  )))                                                                                                                                   
GADM_level1 = GADM_level1.map(lambda f: f.set(ee.Dictionary.fromLists(                                                                
      p := GADM_level3.first().propertyNames().removeAll(GADM_level1.first().propertyNames()),
      ee.List.repeat('NaN', p.size())                                                                                                   
  )))

# Merge the data 
GADM_toSample = GADM_level2.merge(GADM_level0).merge(GADM_level3).merge(GADM_level1)
# print(GADM_toSample)

##/ Define the new regions for Level 3
Pakistan_Gujranwala = ee.FeatureCollection(                        
      ee.Feature(GADM_level3_prep.filterMetadata('NAME_3', 'contains', 'Gujranwala').geometry())
      .set('NAME_0', 'Pakistan')                                                                                                        
      .set('NAME_3', 'Gujranwala')                                                                                                      
      .set('HH7', 'Gujranwala')                                                                                                         
  ).map(lambda f: f.set(ee.Dictionary.fromLists(                                                                                        
      p := GADM_level3_prep.first().propertyNames().removeAll(['NAME_0', 'NAME_3']),                                                    
      ee.List.repeat('NaN', p.size())                                                                                                   
  )))             
                                                                                                                                        
REGIAO_NORTE_OESTE = ee.FeatureCollection(
      ee.Feature(GADM_level2_prep.filterMetadata('NAME_0', 'contains', 'São Tomé and Príncipe')
      .filter(ee.Filter.inList('NAME_2', ['Lembá', 'Lobata'])).geometry())                                                              
      .set('NAME_0', 'São Tomé and Príncipe')
      .set('NAME_2', 'REGIÃO NORTE OESTE')                                                                                              
      .set('HH7', 'REGIAO_NORTE_OESTE')
  ).map(lambda f: f.set(ee.Dictionary.fromLists(                                                                                        
      p := GADM_level2_prep.first().propertyNames().removeAll(['NAME_0', 'NAME_2']),                                                    
      ee.List.repeat('NaN', p.size())
  )))                                                                                                                                   
                  
REGIAO_SUL_ESTE = ee.FeatureCollection(                                                                                               
      ee.Feature(GADM_level2_prep.filterMetadata('NAME_0', 'contains', 'São Tomé and Príncipe')
      .filter(ee.Filter.inList('NAME_2', ['Cantagalo', 'Caué'])).geometry())                                                            
      .set('NAME_0', 'São Tomé and Príncipe')                                                                                           
      .set('NAME_2', 'REGIÃO SUL ESTE')
      .set('HH7', 'REGIAO_SUL_ESTE')                                                                                                    
  ).map(lambda f: f.set(ee.Dictionary.fromLists(                                                                                        
      p := GADM_level2_prep.first().propertyNames().removeAll(['NAME_0', 'NAME_2']),
      ee.List.repeat('NaN', p.size())                                                                                                   
  )))             
                                                                                                                                        
Principe = ee.FeatureCollection(
      ee.Feature(GADM_level1_prep.filterMetadata('NAME_0', 'contains', 'São Tomé and Príncipe')
      .filter(ee.Filter.inList('NAME_1', ['Príncipe'])).geometry())                                                                     
      .set('NAME_0', 'São Tomé and Príncipe')
      .set('NAME_1', 'Príncipe')                                                                                                        
      .set('HH7', 'Príncipe')                                                                                                           
  ).map(lambda f: f.set(ee.Dictionary.fromLists(
      p := GADM_level1_prep.first().propertyNames().removeAll(['NAME_0', 'NAME_1']),                                                    
      ee.List.repeat('NaN', p.size())                                                                                                   
  )))

# Add them to the to sampling data 
GADM_toSample = GADM_toSample.merge(Pakistan_Gujranwala).merge(REGIAO_NORTE_OESTE).merge(REGIAO_SUL_ESTE).merge(Principe)

# filter countries from GADM that are in MICS or DHS datasets
countries_in_DHS = dhs_gdf['CNTRYNAMEE'].unique().tolist()
countries_in_MICS = mics_gdf['CNTRYNAME'].unique().tolist()
countries_to_exclude = set(countries_in_DHS + countries_in_MICS)
GADM_toSample = GADM_toSample.filter(ee.Filter.inList('NAME_0', list(countries_to_exclude)).Not())

NameError: name 'dhs_gdf' is not defined

In [ ]:
# Below list of coutntries that need manual assignment of survey year 
analysisYearByCountry = {
    'Algeria': 2018,
    'Bangladesh': 2019,
    'Central African Republic': 2018,
    'Chad': 2019,
    'Gambia': 2018,
    'Georgia': 2018,
    'Ghana': 2017,
    'Guinea-Bissau': 2018,
    'Guyana': 2019,
    'Iraq': 2018,
    'Kiribati': 2018,
    'Kosovo': 2019,
    'Laos': 2023,
    'Lesotho': 2018,
    'Madagascar': 2018,
    'Mongolia': 2018,
    'Nigeria': 2016,
    'Pakistan': 2017,
    'Palestina': 2019,
    'Paraguay': 2016,
    'Sierra Leone': 2017,
    'Suriname': 2018,
    'São Tomé and Príncipe': 2019,
    'Togo': 2017,
    'Tonga': 2019,
    'Tunisia': 2023,
    'Zimbabwe': 2018,
}

# Get year from dictionary and assign to each feature
year_dict = ee.Dictionary(analysisYearByCountry)                                                                                      
                                                                                                                                    
def assign_year(feature):                                                                                                             
    year = ee.Number(year_dict.get(feature.getString('NAME_0'), ee.Number(2018)))                                                     
    return feature.set('analysis_year', year)                                                                                         
                
GADM_toSample = GADM_toSample.map(assign_year)   

# print countries not in the dictionary
countries_in_GADM = GADM_toSample.aggregate_array('NAME_0').distinct().getInfo()
countries_not_in_dict = set(countries_in_GADM).difference(set(analysisYearByCountry.keys()))
# If empty, print all good, else print the list of countries
if not countries_not_in_dict:
    print("All countries in GADM_toSample have an assigned analysis year.")
else:
    print("The following countries in GADM_toSample do NOT have an assigned analysis year:")
    for country in countries_not_in_dict:
        print(country)

All countries in GADM_toSample have an assigned analysis year.


In [ ]:
# For DHS survey we can use the year "SVYYEAR" and for MICS "SVYYEARS" which define the start year of the survey collection.
# extend analysisYearByCountry to all countries in the DHS and MICS datasets
dhs_country_years = dhs_gdf.groupby('CNTRYNAMEE')['SVYYEAR'].first().apply(int).to_dict()
mics_country_years = mics_gdf.groupby('CNTRYNAME')['SVYYEARS'].first().apply(int).to_dict()

for country in set(dhs_country_years) | set(mics_country_years):
    if country not in analysisYearByCountry:
        if country in dhs_country_years:
            analysisYearByCountry[country] = dhs_country_years[country]
        else:
            analysisYearByCountry[country] = mics_country_years[country]

sortedAnalysisYearByCountry = dict(sorted(analysisYearByCountry.items())) 
for country, year in sortedAnalysisYearByCountry.items():
    print(f"{country}: {year}")


Albania: 2017
Algeria: 2018
Angola: 2023
Armenia: 2016
Bangladesh: 2019
Benin: 2017
Bolivia: 2008
Burkina Faso: 2021
Burundi: 2016
Cambodia: 2021
Cameroon: 2022
Central African Republic: 2018
Chad: 2019
Colombia: 2015
Comoros: 2022
Congo Democratic Republic: 2023
Cote d'Ivoire: 2021
Eswatini: 2021
Ethiopia: 2021
Fiji: 2021
Gabon: 2019
Gambia: 2018
Georgia: 2018
Ghana: 2017
Guatemala: 2015
Guinea: 2021
Guinea-Bissau: 2018
Guyana: 2019
Haiti: 2016
Honduras: 2019
Indonesia: 2017
Iraq: 2018
Jordan: 2023
Kenya: 2022
Kiribati: 2018
Kosovo: 2019
Lao People's Democratic Republic: 2017
Laos: 2023
Lesotho: 2018
Liberia: 2019
Madagascar: 2018
Malawi: 2024
Maldives: 2016
Mali: 2023
Mauritania: 2020
Mongolia: 2018
Mozambique: 2022
Myanmar: 2016
Nepal: 2022
Niger: 2021
Nigeria: 2016
Pakistan: 2017
Palestina: 2019
Paraguay: 2016
Philippines: 2022
Rwanda: 2019
Senegal: 2023
Sierra Leone: 2017
South Africa: 2016
Suriname: 2018
São Tomé and Príncipe: 2019
Tajikistan: 2023
Tanzania: 2022
Thailand: 2022
T

In [ ]:
bandNames = [
    'Abs_Lat',
    'CGIAR_Aridity_Index',
    'CGIAR_PET',
    'CHELSA_BIO_Annual_Mean_Temperature',
    'CHELSA_BIO_Annual_Precipitation',
    'CHELSA_BIO_Precipitation_Seasonality',
    'CHELSA_BIO_Precipitation_of_Coldest_Quarter',
    'CHELSA_BIO_Precipitation_of_Driest_Month',
    'CHELSA_BIO_Precipitation_of_Driest_Quarter',
    'CHELSA_BIO_Precipitation_of_Warmest_Quarter',
    'CHELSA_BIO_Precipitation_of_Wettest_Month',
    'CHELSA_BIO_Precipitation_of_Wettest_Quarter',
    'CHELSA_BIO_Temperature_Annual_Range',
    'CHELSA_BIO_Temperature_Seasonality',
    'CIFOR_TropicalPeatlandExtent',
    'CSP_Global_Human_Modification',
    'ConsensusLandCoverClass_Barren',
    'ConsensusLandCoverClass_Cultivated_and_Managed_Vegetation',
    'ConsensusLandCoverClass_Deciduous_Broadleaf_Trees',
    'ConsensusLandCoverClass_Evergreen_Broadleaf_Trees',
    'ConsensusLandCoverClass_Evergreen_Deciduous_Needleleaf_Trees',
    'ConsensusLandCoverClass_Herbaceous_Vegetation',
    'ConsensusLandCoverClass_Mixed_Other_Trees',
    'ConsensusLandCoverClass_Open_Water',
    'ConsensusLandCoverClass_Regularly_Flooded_Vegetation',
    'ConsensusLandCoverClass_Shrubs',
    'ConsensusLandCoverClass_Snow_Ice',
    'ConsensusLandCoverClass_Urban_Builtup',
    'ConsensusLandCover_Human_Development_Percentage',
    'CrowtherLab_Tree_Density',
    'EarthEnvCloudCover_CloudForestPrediction',
    'EarthEnvTexture_CoOfVar_EVI',
    'EarthEnvTexture_Contrast_EVI',
    'EarthEnvTexture_Correlation_EVI',
    'EarthEnvTexture_Dissimilarity_EVI',
    'EarthEnvTexture_Entropy_EVI',
    'EarthEnvTexture_Evenness_EVI',
    'EarthEnvTexture_Homogeneity_EVI',
    'EarthEnvTexture_Maximum_EVI',
    'EarthEnvTexture_Range_EVI',
    'EarthEnvTexture_Shannon_Index',
    'EarthEnvTexture_Simpson_Index',
    'EarthEnvTexture_Std_EVI',
    'EarthEnvTexture_Uniformity_EVI',
    'EarthEnvTexture_Variance_EVI',
    'EarthEnvTopoMed_1stOrderPartialDerivEW',
    'EarthEnvTopoMed_1stOrderPartialDerivNS',
    'EarthEnvTopoMed_Eastness',
    'EarthEnvTopoMed_Elevation',
    'EarthEnvTopoMed_Roughness',
    'EarthEnvTopoMed_Slope',
    'EarthEnvTopoMed_TerrainRuggednessIndex',
    'EarthEnvTopoMed_TopoPositionIndex',
    'EsaCci_BurntAreasProbability',
    'FanEtAl_Depth_to_Water_Table_AnnualMean',
    'FanEtAl_Depth_to_Water_Table_AnnualSD',
    'GHS_Population_Density',
    'GPWv4_Population_Density',
    'GiriEtAl_MangrovesExtent',
    'GLW3_RuminantsDistribution_downsampled10km',
    'MODIS_EVI',
    'MODIS_NDVI',
    'MODIS_NPP',
    'PelletierEtAl_SoilAndSedimentaryDepositThicknesses',
    'SG_Absolute_depth_to_bedrock',
    'SG_Bulk_density_015cm',
    'SG_Depth_to_bedrock',
    'SG_H2O_Capacity_015cm',
    'SG_Saturated_H2O_Content_015cm',
    'TootchiEtAl_WetlandsRegularlyFlooded',
    'WCS_Human_Footprint_2009']

In [ ]:
# Stack of environmental covariates to sample 
composite = ee.Image.cat([
        ee.Image("projects/crowtherlab/Composite/CrowtherLab_bioComposite_30ArcSec"),
        ee.Image("projects/crowtherlab/Composite/CrowtherLab_climateComposite_30ArcSec"),
        ee.Image("projects/crowtherlab/Composite/CrowtherLab_geoComposite_30ArcSec"),
        ee.Image("projects/crowtherlab/Composite/CrowtherLab_processComposite_30ArcSec"),
        ])

crs = composite.projection().getInfo()['crs']
transform = composite.projection().getInfo()['transform']
print(f"CRS: {crs}")
print(f"Transform: {transform}")

# DYNAMIC COMPOSITE
# Dynamic covariates to sample based on survey year
WorldPop = ee.ImageCollection("projects/sat-io/open-datasets/WORLDPOP/pop")
VIIRS_V21 = ee.ImageCollection("NOAA/VIIRS/DNB/ANNUAL_V21")
VIIRS_V22 = ee.ImageCollection("NOAA/VIIRS/DNB/ANNUAL_V22")
VIIRS_Annual = VIIRS_V21.merge(VIIRS_V22)
CHIRPS_Daily_Precipitation= ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
MODIS_NDVI_EVI = ee.ImageCollection('MODIS/061/MOD13Q1')
JRC_Built_Up = ee.ImageCollection("JRC/GHSL/P2023A/GHS_BUILT_S")
JRC_DegreeUrban = ee.ImageCollection("JRC/GHSL/P2023A/GHS_SMOD_V2-0")
MAP_FrictionSurface = ee.Image("projects/malariaatlasproject/assets/accessibility/friction_surface/2019_v5_1")
ERA5_Monthly_Temp = ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR")

# STATIC COMPOSITE
staticComposite = composite.select(bandNames).addBands(MAP_FrictionSurface.select('friction').rename('map_friction'))


CRS: EPSG:4326
Transform: [0.008333333333333333, 0, -180, 0, -0.008333333333333333, 88]


In [ ]:
# Sample static and dynamic covariates for a single feature
def sampleFeature(feature):
    f = ee.Feature(feature)
    geom = f.geometry()
    reducer = ee.Reducer.mean()

    static_dict = staticComposite.reduceRegion(
        reducer=reducer,
        crs=crs,
        crsTransform=transform,
        geometry=geom,
        maxPixels=1e13,
        tileScale=16
    )

    dynamic_dict = getDynamicImage(ee.Number(f.get('analysis_year'))).reduceRegion(
        reducer=reducer,
        crs=crs,
        crsTransform=transform,
        geometry=geom,
        maxPixels=1e13,
        tileScale=16
    )

    return f.set(static_dict).set(dynamic_dict)


# GHSL built surface: linearly interpolate between 5-year intervals
# 2015-2019 -> between 2015 and 2020
# 2020-2024 -> between 2020 and 2025
def getGHSLBuiltInterpolated(year):
    year = ee.Number(year)

    y0 = year.divide(5).floor().multiply(5).max(2015).min(2020)
    y1 = y0.add(5)
    w = year.subtract(y0).divide(5)

    img0 = ee.Image(
        JRC_Built_Up.filterDate(ee.Date.fromYMD(y0, 1, 1), ee.Date.fromYMD(y0.add(1), 1, 1)).first()
    ).select('built_surface')

    img1 = ee.Image(
        JRC_Built_Up.filterDate(ee.Date.fromYMD(y1, 1, 1), ee.Date.fromYMD(y1.add(1), 1, 1)).first()
    ).select('built_surface')

    return img0.add(img1.subtract(img0).multiply(w)).rename('ghsl_built_surface')

# GHSL Degree of Urbanisation:
# interpolate for yearly estimate, convert class codes to urban mask first
def getGHSLUrbanFraction(year):
    year = ee.Number(year)

    y0 = year.divide(5).floor().multiply(5).max(2015).min(2020)

    smod = ee.Image(
        JRC_DegreeUrban.filterDate(ee.Date.fromYMD(y0, 1, 1), ee.Date.fromYMD(y0.add(1), 1, 1)).first()
    ).select('smod_code')

    return smod.gte(21).rename('ghsl_urban_frac') # classifies 21, 22, 23, and 30 as urban and everything below 21 as non-urban
    
# Annual dynamic composite image 
def getDynamicImage(year):
    year = ee.Number(year).int()

    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, 'year')

    # Returns a fully masked image with given bands — used as fallback when a collection is empty
    def masked_fallback(bands):
        if isinstance(bands, str):
            bands = [bands]
        return ee.Image.constant([0] * len(bands)).rename(bands).updateMask(ee.Image.constant(0))

    # WorldPop: Estimated number of people residing in each grid cell
    worldpop = ee.Image(
        WorldPop.filterDate(start, end)
        .merge(ee.ImageCollection([masked_fallback(['b1'])]))
        .first()
    ).select(0).rename('worldpop')

    # VIIRS annual average radiance (starts 2012; pre-2012 surveys return NaN)
    viirs = ee.Image(
        VIIRS_Annual.filterDate(start, end)
        .merge(ee.ImageCollection([masked_fallback(['average'])]))
        .first()
    ).select('average').rename('viirs_average')

    # CHIRPS annual precipitation sum
    chirps_annual_precipitation = CHIRPS_Daily_Precipitation \
        .filterDate(start, end) \
        .merge(ee.ImageCollection([masked_fallback(['precipitation'])])) \
        .select('precipitation') \
        .sum() \
        .rename('chirps_annual_precipitation')

    # MODIS annual mean NDVI and EVI
    modis = MODIS_NDVI_EVI \
        .filterDate(start, end) \
        .merge(ee.ImageCollection([masked_fallback(['NDVI', 'EVI'])])) \
        .select(['NDVI', 'EVI']) \
        .mean() \
        .rename(['modis_ndvi', 'modis_evi'])

    # GHSL built-up surface (interpolated)
    ghslBuilt = getGHSLBuiltInterpolated(year)

    # GHSL urban fraction
    ghslUrban = getGHSLUrbanFraction(year)

    # ERA5: merge fallback once, reuse for all bands
    era5_with_fallback = ERA5_Monthly_Temp \
        .filterDate(start, end) \
        .merge(ee.ImageCollection([masked_fallback(['temperature_2m', 'runoff_max', 'runoff_min', 'temperature_2m_max'])]))

    era5_temperature_2m = era5_with_fallback \
        .select('temperature_2m').mean().rename('era5_temperature_2m')

    runoff_max_annualmax = era5_with_fallback \
        .select('runoff_max').max().rename('runoff_max_annualmax')

    runoff_min_annualmin = era5_with_fallback \
        .select('runoff_min').min().rename('runoff_min_annualmin')

    temperature_2m_max_annualmax = era5_with_fallback \
        .select('temperature_2m_max').max().rename('temperature_2m_max_annualmax')

    return ee.Image.cat([
        worldpop,
        viirs,
        chirps_annual_precipitation,
        modis,
        ghslBuilt,
        ghslUrban,
        era5_temperature_2m,
        runoff_max_annualmax,
        runoff_min_annualmin,
        temperature_2m_max_annualmax
    ])



In [ ]:
# # Convert DHS and MICS geodataframes to feature collections
# dhs_fc = geemap.geopandas_to_ee(dhs_gdf, geodesic=False)
# mics_fc = geemap.geopandas_to_ee(mics_gdf, geodesic=False)

# export_dhs = ee.batch.Export.table.toAsset(
#     collection=dhs_fc,
#     description='DHS_Survey_Points',
#     assetId='projects/johanvandenhoogen/assets/2026_worldbank/DHS_cleaned'
# )
# export_dhs.start()
# export_mics = ee.batch.Export.table.toAsset(
#     collection=mics_fc,
#     description='MICS_Survey_Points',
#     assetId='projects/johanvandenhoogen/assets/2026_worldbank/MICS_cleaned'
# )
# export_mics.start()

for gdf, path in [(dhs_gdf, 'temp/dhs_fc_cleaned_tmp.shp'),                                                                              
                    (mics_gdf, 'temp/mics_fc_cleaned_tmp.shp')]:                                                                          
    out = gdf.copy()                                                                                                                  
    if 'GEO_ID' in out.columns:                                                                                                       
        out['GEO_ID'] = out['GEO_ID'].astype(str)                                                                                     
    out = out.rename(columns={'analysis_year': 'anl_year'})                                                                           
    out.to_file(path)

# # write to shapefile, if file doesn't exist yet
# if not os.path.exists('temp/dhs_fc_cleaned_tmp.shp'):
#     dhs_gdf.to_file('temp/dhs_fc_cleaned_tmp.shp')
# if not os.path.exists('temp/mics_fc_cleaned_tmp.shp'):
#     mics_gdf.to_file('temp/mics_fc_cleaned_tmp.shp')

# Manual step: upload to GEE 


In [ ]:
dhs_fc = ee.FeatureCollection('projects/johanvandenhoogen/assets/2026_worldbank/dhs_fc_cleaned_tmp')
mics_fc = ee.FeatureCollection('projects/johanvandenhoogen/assets/2026_worldbank/mics_fc_cleaned_tmp')

# rename anl_year back to analysis_year
dhs_fc = dhs_fc.map(lambda f: f.set('analysis_year', f.get('anl_year')))
mics_fc = mics_fc.map(lambda f: f.set('analysis_year', f.get('anl_year')))

# export both to GEE
export_dhs = ee.batch.Export.table.toAsset(
    collection=dhs_fc,
    description='dhs_fc_cleaned',
    assetId='projects/johanvandenhoogen/assets/2026_worldbank/dhs_fc_cleaned'
)
export_dhs.start()  
export_mics = ee.batch.Export.table.toAsset(
    collection=mics_fc,
    description='mics_fc_cleaned',
    assetId='projects/johanvandenhoogen/assets/2026_worldbank/mics_fc_cleaned'
)
export_mics.start()

In [ ]:
dhs_fc = ee.FeatureCollection('projects/johanvandenhoogen/assets/2026_worldbank/dhs_fc_cleaned')
mics_fc = ee.FeatureCollection('projects/johanvandenhoogen/assets/2026_worldbank/mics_fc_cleaned')

# Final dataset to sample: GADM_toSample, dhs_fc, and mics_fc merged
final_fc = ee.FeatureCollection([GADM_toSample, dhs_fc, mics_fc]).flatten()
print(f"Total number of features to sample: {final_fc.size().getInfo()}")

Total number of features to sample: 1302


In [ ]:
# Check for non-numeric values that wouldn't show as NaN                                                                              
from numpy import sort


print(dhs_gdf['analysis_year'].dtype)                                                                                                 
print(dhs_gdf['analysis_year'].unique()[:20])        

print(mics_gdf['analysis_year'].dtype)                                                                                                 
print(mics_gdf['analysis_year'].unique()[:20])
                                                                                                                                    
# Check GADM features server-side (sample 10)                                                                                         
print(GADM_toSample.aggregate_histogram('analysis_year').keys().getInfo())
print(sorted(dhs_fc.aggregate_histogram('analysis_year').keys().getInfo()))
print(sorted(mics_fc.aggregate_histogram('analysis_year').keys().getInfo()))

int64
[2023 2016 2019 2015 2020 2017 2022 2021 2024 2014 2018 2008]
int64
[2018 2022 2017 2019 2016 2021]
['2016', '2018', '2019', '2023']
['2008', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']
['2016', '2017', '2018', '2019', '2021', '2022']


In [ ]:
# Check country/year/level combinations across all sources
# Admin level is inferred from the highest NAME_N property that is set and not 'NaN'
def infer_level(feature):
    def is_valid(prop):
        val = ee.String(ee.Algorithms.If(
            feature.propertyNames().contains(prop),
            feature.get(prop), 'NaN'
        ))
        return val.compareTo('NaN').neq(0).And(val.length().gt(0))
    return ee.Number(ee.Algorithms.If(is_valid('NAME_3'), 3,
                     ee.Algorithms.If(is_valid('NAME_2'), 2,
                     ee.Algorithms.If(is_valid('NAME_1'), 1, 0))))

def gadm_row(feature):
    return ee.Feature(None, {
        'country': feature.getString('NAME_0'),
        'year':    ee.Number(feature.get('analysis_year')).int(),
        'level':   infer_level(feature),
        'region':  feature.getString('HH7')
    })

def dhs_row(feature):
    return ee.Feature(None, {
        'country': feature.getString('CNTRYNAMEE'),
        'year':    ee.Number(feature.get('analysis_year')).int(),
        'level':   ee.String('DHS'),
        'region':  feature.getString('DHSREGEN')
    })

def mics_row(feature):
    return ee.Feature(None, {
        'country': feature.getString('CNTRYNAME'),
        'year':    ee.Number(feature.get('analysis_year')).int(),
        'level':   ee.String('MICS'),
        'region':  feature.getString('REGNAME')
    })

country_years = ee.FeatureCollection([
    GADM_toSample.map(gadm_row).distinct(['country', 'year', 'level']),
    dhs_fc.map(dhs_row).distinct(['country', 'year', 'level']),
    mics_fc.map(mics_row).distinct(['country', 'year', 'level']),
]).flatten()

df = geemap.ee_to_df(country_years) \
          .sort_values(['level', 'country']) \
          .reset_index(drop=True)

# Flag countries whose year doesn't match analysisYearByCountry
df['expected_year'] = df['country'].map(analysisYearByCountry)
df['ok'] = df.apply(lambda r: r['year'] == r['expected_year'] or pd.isna(r['expected_year']), axis=1)
print(df[['level','country','year','expected_year','ok']].to_string())


   level                           country  year  expected_year     ok
0      1                           Algeria  2018           2018   True
1      1          Central African Republic  2018           2018   True
2      1                     Guinea-Bissau  2018           2018   True
3      1                            Guyana  2019           2019   True
4      1                              Iraq  2018           2018   True
5      1                          Kiribati  2018           2018   True
6      1                              Laos  2023           2023   True
7      1                          Mongolia  2018           2018   True
8      1                          Paraguay  2016           2016   True
9      1                          Suriname  2018           2018   True
10     1             São Tomé and Príncipe  2019           2019   True
11     1                             Tonga  2019           2019   True
12     1                           Tunisia  2023           2023   True
13    

In [ ]:
# Sample variables
worldbank_sampled = final_fc.map(sampleFeature)

export_task = ee.batch.Export.table.toAsset(
    collection = worldbank_sampled,
    description = 'worldbank_sampled',
    assetId = 'projects/johanvandenhoogen/assets/2026_worldbank/worldbank_sampled'
)
export_task.start()

In [ ]:
# After export is finished, load the sampled data as a feature collection and convert to geodataframe
worldbank_sampled_fc = ee.FeatureCollection('projects/johanvandenhoogen/assets/2026_worldbank/worldbank_sampled')
worldbank_sampled_gdf = geemap.ee_to_gdf(worldbank_sampled_fc)
worldbank_sampled_gdf.head()
worldbank_sampled_gdf.to_csv('data/worldbank_sampled.csv', index=False)

In [ ]:
worldbank_sampled_gdf.head()


,geometry,Abs_Lat,CC_1,CC_2,CC_3,CGIAR_Aridity_Index,CGIAR_PET,CHELSA_BIO_Annual_Mean_Temperature,CHELSA_BIO_Annual_Precipitation,CHELSA_BIO_Precipitation_Seasonality,...,era5_temperature_2m,ghsl_built_surface,ghsl_urban_frac,map_friction,modis_evi,modis_ndvi,runoff_max_annualmax,runoff_min_annualmin,temperature_2m_max_annualmax,viirs_average
0,"MULTIPOLYGON (((43.62283 -12.28618, 43.62306 -...",12.323293,None,None,None,10309.259542,2007.328500,250.282019,1342.349195,67.906132,...,298.924320,1.982414,0.189731,0.037722,4664.112468,6795.885880,0.007733,-1.490116e-08,302.925232,0.268968
1,"MULTIPOLYGON (((44.19802 -12.16507, 44.19807 -...",12.222260,None,None,None,9394.301269,1978.478524,237.320639,1955.272813,67.012768,...,297.613088,40.870576,0.039374,0.043836,4191.513321,6537.058973,0.003740,1.142953e-06,304.112908,0.296196
2,"POLYGON ((43.22855 -11.73635, 43.22877 -11.736...",11.662888,None,None,None,12717.777366,1829.624897,226.023979,1787.283132,55.089772,...,296.523902,33.855743,0.176459,0.036054,4021.184823,6614.508839,0.002627,8.889113e-07,302.813624,0.351568
3,"MULTIPOLYGON (((43.62283 -12.28618, 43.62306 -...",12.323293,None,None,None,10309.259542,2007.328500,250.282019,1342.349195,67.906132,...,298.924320,1.982414,0.189731,0.037722,4664.112468,6795.885880,0.007733,-1.490116e-08,302.925232,0.268968
4,"MULTIPOLYGON (((44.19802 -12.16507, 44.19807 -...",12.222260,None,None,None,9394.301269,1978.478524,237.320639,1955.272813,67.012768,...,297.613088,40.870576,0.039374,0.043836,4191.513321,6537.058973,0.003740,1.142953e-06,304.112908,0.296196


In [ ]:
# Coverage check: tally country / source / level / year from worldbank_sampled_gdf

def _nonempty(r, col):
    v = r.get(col, '')
    return pd.notna(v) and str(v).strip() not in ('', 'nan', 'NaN')

def classify_row(r):
    if _nonempty(r, 'CNTRYNAMEE'):
        return str(r['CNTRYNAMEE']), 'DHS', '1'
    if _nonempty(r, 'CNTRYNAME'):
        if _nonempty(r, 'GEONAMET'):   lvl = '3'
        elif _nonempty(r, 'GEONAMES'): lvl = '2'
        elif _nonempty(r, 'GEONAME'):  lvl = '1'
        else:                           lvl = '0'
        return str(r['CNTRYNAME']), 'MICS', lvl
    country = str(r.get('NAME_0', ''))
    if _nonempty(r, 'NAME_3'):   lvl = '3'
    elif _nonempty(r, 'NAME_2'): lvl = '2'
    elif _nonempty(r, 'NAME_1'): lvl = '1'
    else:                         lvl = '0'
    return country, 'GADM', lvl

worldbank_sampled_gdf[['country','source','level']] = pd.DataFrame(
    worldbank_sampled_gdf.apply(classify_row, axis=1).tolist(), index=worldbank_sampled_gdf.index
)

coverage_df = (
    worldbank_sampled_gdf.groupby(['country','source','level'])['analysis_year']
    .apply(lambda s: ', '.join(str(int(y)) for y in sorted(s.dropna().unique())))
    .reset_index()
    .rename(columns={'analysis_year': 'years'})
    .sort_values(['source','country','level'])
    .reset_index(drop=True)
)
print(coverage_df.to_string())
coverage_df.to_csv('worldbank_sampled_coverage.csv', index=False)

                              country source level       years
0                             Albania    DHS     1        2017
1                              Angola    DHS     1        2023
2                             Armenia    DHS     1        2016
3                          Bangladesh    DHS     1        2022
4                               Benin    DHS     1        2017
5                             Bolivia    DHS     1        2008
6                        Burkina Faso    DHS     1        2021
7                             Burundi    DHS     1        2016
8                            Cambodia    DHS     1        2021
9                            Cameroon    DHS     1        2022
10                               Chad    DHS     1        2014
11                           Colombia    DHS     1        2015
12          Congo Democratic Republic    DHS     1        2023
13                      Cote d'Ivoire    DHS     1        2021
14                           Ethiopia    DHS     1  201